# Grounding DINO tiny — DIMER zero-shot object detection tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/grounding-dino-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/grounding-dino-detection-pipeline/blob/main/tutorials/grounding_dino_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-IDEA--Research%2Fgrounding--dino--tiny-ffcc4d?style=flat)](https://huggingface.co/IDEA-Research/grounding-dino-tiny) [![Upstream](https://img.shields.io/badge/Upstream-IDEA--Research%2FGroundingDINO-181717?style=flat&logo=github&logoColor=white)](https://github.com/IDEA-Research/GroundingDINO) [![arXiv](https://img.shields.io/badge/arXiv-2303.05499-b31b1b.svg)](https://arxiv.org/abs/2303.05499)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** zero-shot (open-vocabulary, text-prompted) object detection using the pinned `IDEA-Research/grounding-dino-tiny` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/grounding_dino_detection_pipeline/pipeline.py` at revision `81d1883de6f3`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a2bb814dd30d776dcf7e30523b00659f4f141c71` (~690 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the image is resized (shortest edge 800, longest edge 1333) and the caller's phrases are joined into one lower-cased, period-separated text (`"rectangle. red circle."`); a Swin-T image backbone and a BERT text encoder are fused, and the decoder proposes boxes whose per-token grounding scores pass a **sigmoid**. The pipeline returns each surviving box in xyxy pixel coordinates of the input image, the phrase it grounded to, and its score. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried pipeline module adds snapshot verification, prompt and image validation with named ceilings, a fixed output contract and the `box_iou`, `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic scene drawn in code; the IoU values reported for it are sanity evidence against the boxes you drew, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with known object boxes, validate the image and the prompts into an input manifest through the pipeline's own validation stage, run text-prompted detection through the public API with explicit caller-owned thresholds, read sigmoid scores and thresholds correctly, produce an evaluation report that is `sample-sanity` with `box_iou` only when reference boxes exist and `not-measurable` otherwise, exercise an optional BYOD path, and export machine-readable detections plus an annotated image and provenance.

**This notebook does not demonstrate:** instance or semantic segmentation (see the sibling SAM 2 pipeline), tracking, OCR, captioning, closed-set detection with a fixed class list, mAP or precision/recall evaluation (which needs a labelled box set), or any training. Prompts are free text, so a phrase the model cannot ground still produces boxes for something — the score, not the label, is your only signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate for one small image: the repository's model card records 6.2 s to load and 4.4 s per `detect` on the 320×240 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the ~689 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures.
- **Data:** the default sample is a deterministic 320×240 scene drawn in code (grey background, one dark rectangle, one red disc) with two prompts naming them, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own comma-separated prompt phrases (1–16 phrases, at most 48 characters each). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `IDEA-Research/grounding-dino-tiny` snapshot (~690 MB in total) at revision `a2bb814dd30d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'grounding-dino-detection-pipeline',
    'repository_revision': '81d1883de6f3dbeb0adcd135518f48d8916d7539',
    'embedded_module': 'src/grounding_dino_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/grounding_dino_detection_pipeline/pipeline.py'],
    'module_sha256': 'a919ad96aca0a1934f765ba92558f0d1405b96e63ef3e241a29bb7459da29175',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/grounding_dino_detection_pipeline/` @ `81d1883de6f3`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/grounding_dino_detection_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "IDEA-Research/grounding-dino-tiny"
MODEL_REVISION = "a2bb814dd30d776dcf7e30523b00659f4f141c71"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "grounding-dino-tiny"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Thresholds: the values the upstream model card's usage example passes to
# post_process_grounded_object_detection (box_threshold=0.4, text_threshold=0.3). Both gate a sigmoid
# score that is not calibrated; the deployment owns tuning them on its own labelled data.
BOX_THRESHOLD = 0.4
TEXT_THRESHOLD = 0.3
# Input ceilings. The processor resizes to shortest edge 800 / longest edge 1333 (preprocessor_config.json),
# so image cost is bounded; the text side is bounded by config.json "max_text_len": 256 tokens.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 48
MAX_TEXT_TOKENS = 256


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def format_prompts(prompts: Sequence[str]) -> str:
    """Validate a list of phrases and join them the way the upstream card instructs: lowercase, each
    phrase terminated by a period, separated by a space ("a cat. a remote control.")."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence):
        raise TypeError("prompts must be a list of phrases, not a single string")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
    cleaned: list[str] = []
    for phrase in prompts:
        if not isinstance(phrase, str):
            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")
        text = phrase.strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("prompt phrases must not be empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(
                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"
            )
        cleaned.append(text)
    return " ".join(f"{text}." for text in cleaned)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prompts": [1, MAX_PROMPTS],
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "prompt_tokens": [1, MAX_TEXT_TOKENS],
    "box_threshold": [0.0, 1.0],
    "text_threshold": [0.0, 1.0],
    "preprocessing": (
        "image converted to RGB; phrases stripped, lower-cased, period-terminated and space-joined "
        "(format_prompts); the processor resizes to shortest edge 800 / longest edge 1333 and returned "
        "boxes are mapped back to input pixels"
    ),
}


def _check_inputs(
    image: Any, prompts: Any, box_threshold: Any, text_threshold: Any
) -> tuple[Image.Image, str, float, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    text = format_prompts(prompts)
    box_t = _check_threshold("box_threshold", box_threshold)
    text_t = _check_threshold("text_threshold", text_threshold)
    return rgb, text, box_t, text_t


def validate_inputs(
    image: Image.Image,
    prompts: Sequence[str],
    *,
    box_threshold: float = BOX_THRESHOLD,
    text_threshold: float = TEXT_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, text, box_t, text_t = _check_inputs(image, prompts, box_threshold, text_threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_prompts": len(prompts),
            }
        ],
        "prompt_text": text,
        "box_threshold": box_t,
        "text_threshold": text_t,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[float]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (phrase -> xyxy reference box) the report carries one ``box_iou``
    entry per reference as sample-sanity geometry evidence; without them the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "zero-shot (open-vocabulary, text-prompted) object detection",
        "decision_rule": (
            "a box survives when its best grounding score reaches box_threshold and its phrase "
            "tokens reach text_threshold; the score is an uncalibrated sigmoid, not a probability"
        ),
        "box_threshold": result.get("box_threshold", BOX_THRESHOLD),
        "text_threshold": result.get("text_threshold", TEXT_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes on your own images with a phrase vocabulary matching the prompts, "
                "scored per object with box_iou and aggregated into precision/recall or mean average "
                "precision at a stated IoU threshold; no such labelled set ships with this repository"
            ),
        }
    metrics = []
    for phrase, box in ground_truth_boxes.items():
        ious = [box_iou(det["box"], box) for det in detections]
        best = max(range(len(ious)), key=ious.__getitem__) if ious else None
        metrics.append(
            {
                "id": "box_iou",
                "reference": phrase,
                "value": ious[best] if best is not None else 0.0,
                "matched_label": detections[best]["label"] if best is not None else None,
                "label_matches_reference": (detections[best]["label"] == phrase)
                if best is not None
                else False,
                "estimation": "one reference box per phrase on a single scene, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial sample; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled box set from the deployment domain with a matching phrase vocabulary for any "
            "mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class GroundingDINOPipeline:
    """Text-prompted (open-vocabulary) object detection over the pinned Grounding DINO tiny checkpoint."""

    _runner: Callable[[Image.Image, str, float, float], list[dict[str, Any]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GroundingDINOPipeline:
        import torch
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        processor = AutoProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = AutoModelForZeroShotObjectDetection.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image, text: str, box_threshold: float, text_threshold: float) -> list[dict]:
            inputs = processor(images=image, text=text, return_tensors="pt")
            n_tokens = int(inputs["input_ids"].shape[1])
            if n_tokens > MAX_TEXT_TOKENS:
                raise ValueError(f"prompt text is {n_tokens} tokens > MAX_TEXT_TOKENS {MAX_TEXT_TOKENS}")
            inputs = inputs.to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            result = processor.post_process_grounded_object_detection(
                outputs,
                inputs["input_ids"],
                threshold=box_threshold,
                text_threshold=text_threshold,
                target_sizes=[image.size[::-1]],
            )[0]
            labels = result.get("text_labels") or result.get("labels")
            return [
                {"box": [float(v) for v in box.tolist()], "label": str(label), "score": float(score)}
                for box, label, score in zip(result["boxes"], labels, result["scores"], strict=True)
            ]

        return cls(runner, resolved_device)

    def detect(
        self,
        image: Image.Image,
        prompts: Sequence[str],
        *,
        box_threshold: float = BOX_THRESHOLD,
        text_threshold: float = TEXT_THRESHOLD,
    ) -> dict[str, Any]:
        """Detect the phrases in `prompts`; boxes are xyxy pixel coordinates in the input image."""
        rgb, text, box_t, text_t = _check_inputs(image, prompts, box_threshold, text_threshold)
        detections = self._runner(rgb, text, box_t, text_t)
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "prompt_text": text,
            "box_threshold": box_t,
            "text_threshold": text_t,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a2bb814dd30d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GroundingDINOPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "grounding-dino-tiny",
  "modelId": "IDEA-Research/grounding-dino-tiny",
  "revision": "a2bb814dd30d776dcf7e30523b00659f4f141c71",
  "files": [
    {
      "path": "README.md",
      "bytes": 2580,
      "sha256": "cf46f74c7b6850f1d5cbe406028324d8798148726016d46a69a365b4a2d3e89f"
    },
    {
      "path": "added_tokens.json",
      "bytes": 82,
      "sha256": "909e96cb32d92ce728a01bc99850cbba26196d74115c17ebeb019275412588f2"
    },
    {
      "path": "config.json",
      "bytes": 1644,
      "sha256": "eec82c5ab66e16df12a9a212e68ac011779927c2536cf9078658e35d85f0c67a"
    },
    {
      "path": "model.safetensors",
      "bytes": 689359096,
      "sha256": "1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 457,
      "sha256": "8454179ba95e2ad22947835aad7b45862a601fc0055ab88bf1ee70892d3aea60"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1237,
      "sha256": "d40ab645b68211910b9170d22433d43186a6ec8ee6fd10ba170524b25bf4fb56"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 690308125
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = GroundingDINOPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: a deterministic 320×240 RGB scene is drawn in code — grey background, a dark filled rectangle at `[40, 60, 140, 180]` and a red filled disc whose bounding box is `[200, 80, 280, 160]` — and the prompts name them (`rectangle`, `red circle`). This is the same scene and prompt pair the repository's smoke run used. The drawn boxes are the reference for the `box_iou` sanity check later; they are not a labelled dataset, so nothing here is a precision/recall measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image and set `BYOD_PROMPTS` to the phrases you want grounded — no reference boxes exist for it, so the evaluation report will be `not-measurable`.

The detection thresholds are **caller-owned request parameters**, not pipeline constants: `box_threshold` keeps a box whose best grounding score reaches it, `text_threshold` keeps the phrase tokens that reach it when the box is labelled. Their package defaults (`BOX_THRESHOLD = 0.4`, `TEXT_THRESHOLD = 0.3`) follow the upstream card's usage example, not a calibration; they are exposed here as form parameters and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image, the prompts and both thresholds to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the prompts, the thresholds, and the drawn reference boxes.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PROMPTS = 'cat, remote control'  # @param {type:"string"}
box_threshold = 0.4  # @param {type:"number"}
text_threshold = 0.3  # @param {type:"number"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    prompts = [phrase.strip() for phrase in BYOD_PROMPTS.split(',') if phrase.strip()]
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    image = Image.new('RGB', (320, 240), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    drawn_boxes = {'rectangle': [40.0, 60.0, 140.0, 180.0], 'red circle': [200.0, 80.0, 280.0, 160.0]}
    draw.rectangle(drawn_boxes['rectangle'], fill=(30, 30, 30))
    draw.ellipse(drawn_boxes['red circle'], fill=(220, 30, 30))
    prompts = list(drawn_boxes)
    image_name = 'synthetic_scene_320x240.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'prompts': prompts, 'box_threshold': box_threshold, 'text_threshold': text_threshold, 'drawn_boxes': drawn_boxes})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `detect` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, 1..`MAX_PROMPTS` phrases of at most `MAX_PROMPT_CHARS` characters each, and both thresholds in `[0, 1]` — canonicalises the phrases through the package's `format_prompts` (lower-cased, period-terminated, space-joined) and returns an **input manifest** naming the schema and ceilings, the input's observed mode and size, the prompt text actually sent to the tokenizer, the thresholds, and the verdict. The manifest is written to `outputs/grounding_dino_detection_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately over-long phrase and records the pipeline's own error message as a finding. `MAX_TEXT_TOKENS` (256) is enforced later, inside the pipeline, because it counts tokenizer output rather than characters. Inside the pipeline the image is converted to RGB and resized by the processor; boxes are mapped back to input pixels, and nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS}})
input_manifest = validate_inputs(image, prompts, box_threshold=box_threshold, text_threshold=text_threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, ['x' * (MAX_PROMPT_CHARS + 1)])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-prompt-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/grounding_dino_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Detect and read the scores correctly

`detect` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` the grounded phrase text — plus `prompt_text`, the thresholds used, `width`, `height` and the model identity. Each `score` is a **sigmoid grounding score, not a calibrated probability**: it was never fitted to the frequency with which a box is correct, so 0.9 does not mean "90 % likely". The thresholds you passed are the only decision rule; the pipeline ships them as defaults, not as a calibration, and the caller owns them per deployment — raise `box_threshold` when false boxes cost more than missed ones, lower it for recall, and note that a phrase that grounds nothing well may still surface a box just above the threshold. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place and reorder near-ties. As recorded in the model card, the repository's CPU smoke on this same scene at the default thresholds returned `red circle` at score 0.932 and `rectangle` at 0.698 with boxes within about 2 px of the drawn ones; that is one observation, not a calibration point. A materially different result on your runtime is a signal to check the install, not a measurement.

In [ ]:
result = pipe.detect(image, prompts, box_threshold=box_threshold, text_threshold=text_threshold)
print({'n_detections': len(result['detections']), 'prompt_text': result['prompt_text'], 'box_threshold': result['box_threshold'], 'text_threshold': result['text_threshold'], 'device': pipe.device})
for rank, det in enumerate(result['detections'], start=1):
    print(f"{rank:>2}. score {det['score']:.4f}  label {det['label']!r}  box {[round(v, 1) for v in det['box']]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No detection metric is reported by default: mean average precision needs a labelled box set with a matching vocabulary, and this repository ships none. The repository's only metric helper is `box_iou(a, b)` (intersection-over-union of two xyxy boxes), the building block a caller would use to compute mAP on their own labelled data; when reference boxes are supplied the report carries one `box_iou` entry per reference — its value, which detection matched it best and whether that detection's label agrees — with the verdict `sample-sanity`. On the synthetic path those references are shapes **you drew yourself**, so a high IoU proves only that the input contract, prompt formatting, forward pass and coordinate mapping round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable: labelled boxes on your own images with a phrase vocabulary matching the prompts. The report is written to `outputs/grounding_dino_detection_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, drawn_boxes, sample_kind=sample_kind)
with open('outputs/grounding_dino_detection_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No reference boxes exist for this input, so box_iou is not computed; inspect the annotated PNG instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered detections with boxes and labels, prompt text, thresholds), the evaluation report, the input manifest, the sample identity, digest and drawn boxes, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The detections are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns so score ordering survives downstream use, and an annotated PNG draws every returned box for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for det in result['detections']:
    draw.rectangle(det['box'], outline=(0, 255, 0), width=2)
    draw.text((det['box'][0] + 2, det['box'][1] + 2), f"{det['label']} {det['score']:.2f}", fill=(0, 255, 0))
annotated.save('outputs/grounding_dino_detection_annotated.png')
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'prompts': prompts, 'drawn_boxes': drawn_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/grounding_dino_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/grounding_dino_detection_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes are grounded to free-text phrases: the label tells you which phrase the box scored best against, not that the object is really there, and the sigmoid score is uncalibrated. The thresholds are request parameters you own; the defaults are the upstream usage example, not a tuned operating point. On the synthetic scene the `box_iou` values in the evaluation report compare detections to shapes you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, prompt formatting, forward pass and coordinate mapping work; they say nothing about photographs, small or occluded objects, crowded scenes, or vocabulary the model has never grounded, and a BYOD result is a single-image observation with the verdict `not-measurable`. Long or many prompts are refused at the stated ceilings, and phrases are lower-cased and period-joined before encoding, which can merge or split labels in ways you should inspect in `prompt_text`. The pipeline provides no segmentation, tracking, OCR, captioning, mAP evaluation, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `box_threshold` to 0.2 and count how many extra boxes appear on the same scene (the smoke run found none, but a photograph behaves differently); add a phrase that names nothing in the image (`"bicycle"`) and watch where its box lands and how its score compares; enable `USE_BYOD` with a photograph, hand-label a few boxes and pass them to `evaluation_report` to see the verdict switch to `sample-sanity` — the first step towards a real precision/recall number.

## References

- Repository README: https://github.com/kurtvalcorza/grounding-dino-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/grounding-dino-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/grounding-dino-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/IDEA-Research/grounding-dino-tiny
- Upstream code: https://github.com/IDEA-Research/GroundingDINO
- Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection (Liu et al., 2023): https://arxiv.org/abs/2303.05499